# COLTIE Immersion · Week 1 · **Monday — Python & Data Fundamentals**

You already know Python. Today is not about syntax, we move fast through idioms, push hard on **NumPy vectorization** and **intermediate pandas**, and end by building a **reusable cleaning pipeline** on a synthetic transportation dataset.

**By the end of today you will have:**
- A fast, *vectorized* mental model — no looping over rows
- Comfort with intermediate pandas: chaining, `groupby` (`agg`/`transform`/`apply`), vectorized `.dt`/`.str`, `category` dtype
- A typed, reusable **`clean_crashes()` pipeline** + a clean summary table — the artifact you commit on Friday

**The dataset:** synthetic Iowa crash records (intentionally messy).

> **One rule for the whole bootcamp:** if you find yourself writing a `for` loop over DataFrame rows, stop — there is almost always a vectorized way, and it is usually 50–100× faster.

## 0 · Setup & the dataset

Run this cell first. It generates `crashes_raw.csv` so the notebook is fully self-contained — no external downloads.

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from collections import Counter, defaultdict
from pathlib import Path
from typing import Callable, Iterable
import itertools, functools, time

# One seed for the whole notebook → reproducible runs (a research non-negotiable)
SEED = 42
rng = np.random.default_rng(SEED)

def generate_raw_crashes(n: int = 1200, seed: int = SEED) -> pd.DataFrame:
    """Synthetic Iowa crash records with realistic, *intentional* data-quality issues."""
    g = np.random.default_rng(seed)
    counties  = np.array(["Story", "Polk", "Linn", "Scott", "Johnson", "Black Hawk"])
    routes    = np.array(["I-35", "I-80", "US-30", "US-69", "IA-210", "IA-1"])
    weather   = np.array(["Clear", "Rain", "Snow", "Fog"])
    lighting  = np.array(["Daylight", "Dark", "Dusk", "Dawn"])
    severity  = np.array(["No Injury", "Minor", "Major", "Fatal"])

    # time-of-day structure: ~45% of crashes cluster in rush hours
    start  = pd.Timestamp("2024-01-01")
    day    = g.integers(0, 365, n)
    rush   = g.random(n) < 0.45
    hour   = np.where(rush, g.choice([7, 8, 16, 17, 18], n), g.integers(0, 24, n))
    minute = g.integers(0, 60, n)
    dt = start + pd.to_timedelta(day, "D") + pd.to_timedelta(hour, "h") + pd.to_timedelta(minute, "m")

    spd_limit = g.choice([25, 35, 45, 55, 65, 70], n)
    veh_speed = np.clip(spd_limit + g.normal(3, 9, n), 0, None).round(0)
    light     = g.choice(lighting, n, p=[0.55, 0.30, 0.08, 0.07])
    ww_prob   = 0.02 + 0.05 * (light == "Dark")          # wrong-way is rare, worse in the dark

    df = pd.DataFrame({
        "crash_id":       [f"IA-2024-{i:05d}" for i in range(n)],
        "crash_datetime": dt,
        "county":         g.choice(counties, n),
        "route":          g.choice(routes, n),
        "severity":       g.choice(severity, n, p=[0.62, 0.28, 0.085, 0.015]),  # imbalanced
        "num_vehicles":   g.integers(1, 5, n),
        "weather":        g.choice(weather, n, p=[0.60, 0.25, 0.10, 0.05]),
        "lighting":       light,
        "speed_limit":    spd_limit,
        "vehicle_speed":  veh_speed,
        "wrong_way":      g.random(n) < ww_prob,
        "latitude":       g.uniform(40.5, 43.3, n).round(5),
        "longitude":      g.uniform(-96.5, -90.2, n).round(5),
    })

    # --- inject realistic messiness ---
    up = g.random(n) < 0.15;  df.loc[up, "county"]  = df.loc[up, "county"].str.upper()          # casing
    sp = g.random(n) < 0.10;  df.loc[sp, "county"]  = df.loc[sp, "county"] + " "                 # trailing space
    lo = g.random(n) < 0.12;  df.loc[lo, "weather"] = df.loc[lo, "weather"].str.lower()          # casing
    for col, frac in [("weather", .06), ("lighting", .05), ("speed_limit", .04),
                      ("latitude", .03), ("longitude", .03), ("crash_datetime", .02)]:
        df.loc[g.random(n) < frac, col] = np.nan                                                 # missing
    df.loc[g.random(n) < 0.04, "num_vehicles"]  = 999                                            # sentinel
    df.loc[g.random(n) < 0.02, "vehicle_speed"] = -1                                             # impossible
    df.loc[g.random(n) < 0.01, "vehicle_speed"] = 250                                            # outlier
    df = pd.concat([df, df.sample(15, random_state=0)], ignore_index=True)                       # dupes
    return df

generate_raw_crashes().to_csv("crashes_raw.csv", index=False)
print("Wrote crashes_raw.csv")

Wrote crashes_raw.csv


**What's wrong with this data — on purpose.** Real transportation data is never clean, so neither is this:

- **Inconsistent categories** — `Story`, `STORY`, `"Story "` are the same county typed three ways
- **Missing values** scattered across weather, lighting, speed, coordinates, and timestamps
- **Sentinel values** — `num_vehicles == 999` means "unknown", not 999 cars
- **Impossible / outlier speeds** — `-1` and `250` mph
- **Duplicate rows**, and a **class-imbalanced** target (`severity`, `wrong_way` are rare)

Your job in the lab is a pipeline that handles every one of these — *vectorized*, *documented*, *reproducible*.

## 1 · Pythonic idioms for research

Readable, intentional Python = fewer bugs and reviewable code. A quick pass through the patterns you'll reach for daily.

In [ ]:
# Comprehensions — nested + conditional. Build a lookup without a single append().
routes  = ["I-35", "I-80", "US-30", "IA-210"]
hours   = [8, 17, 2]
# (route, hour) pairs only for highway routes, tagged rush/off-peak
pairs = [(r, h, "rush" if h in (7, 8, 16, 17, 18) else "off-peak")
         for r in routes if r.startswith("I-")
         for h in hours]
pairs

[('I-35', 8, 'rush'),
 ('I-35', 17, 'rush'),
 ('I-35', 2, 'off-peak'),
 ('I-80', 8, 'rush'),
 ('I-80', 17, 'rush'),
 ('I-80', 2, 'off-peak')]

In [ ]:
# Generators — process data you don't want to hold in memory all at once.
# A generator FUNCTION streams one row at a time (think: a 50 GB log file).
def stream_lines(path: str) -> Iterable[str]:
    with open(path) as fh:          # context manager → file always closes, even on error
        for line in fh:
            yield line.rstrip("\n")

first3 = list(itertools.islice(stream_lines("crashes_raw.csv"), 3))

# A generator EXPRESSION (parentheses, not brackets) — lazy, no intermediate list
n_header_fields = sum(1 for _ in first3[0].split(","))
print("rows peeked:", len(first3), "| columns in header:", n_header_fields)

rows peeked: 3 | columns in header: 13


In [ ]:
# Standard-library power tools — reach for these before writing manual bookkeeping.
sample = ["Clear", "Rain", "Clear", "Snow", "Rain", "Clear"]

print("Counter:", Counter(sample).most_common())     # frequency table in one line

by_first = defaultdict(list)                          # no KeyError, no setdefault dance
for r in routes:
    by_first[r[0]].append(r)
print("defaultdict:", dict(by_first))

# zip + enumerate + tuple unpacking
for i, (route, hour, tag) in enumerate(pairs[:3]):
    print(f"  [{i}] {route} @ {hour:02d}:00 → {tag}")

Counter: [('Clear', 3), ('Rain', 2), ('Snow', 1)]
defaultdict: {'I': ['I-35', 'I-80', 'IA-210'], 'U': ['US-30']}
  [0] I-35 @ 08:00 → rush
  [1] I-35 @ 17:00 → rush
  [2] I-35 @ 02:00 → off-peak


In [ ]:
# pathlib + dataclass config + type hints + try/except — a tiny, robust loader.
@dataclass
class LoadConfig:
    path: Path = Path("crashes_raw.csv")
    parse_dates: tuple[str, ...] = ("crash_datetime",)

def safe_row_count(cfg: LoadConfig) -> int:
    """Return the number of data rows, or -1 if the file is missing/unreadable."""
    try:
        with cfg.path.open() as fh:
            return sum(1 for _ in fh) - 1          # minus the header
    except (FileNotFoundError, OSError) as e:
        print(f"could not read {cfg.path}: {e}")
        return -1

safe_row_count(LoadConfig())

1215

**Exercise 1.** Using a **single comprehension**, build a list of the `crash_id`s that are *odd-numbered*
(e.g. `IA-2024-00001`, `IA-2024-00003`, …) for the first 20 crashes. Then count how many there are with `len()`.
*Hint: the trailing digits are `cid.split("-")[-1]`.*

In [ ]:
# Your turn — Exercise 1
first20 = list(itertools.islice(stream_lines("crashes_raw.csv"), 1, 21))

odd_cids = [
    line.split(",")[0]
    for line in first20
    if int(line.split(",")[0].split("-")[-1]) % 2 != 0
]

print("Odd Crash IDs: ", odd_cids)
print("Count: ", len(odd_cids))

Odd Crash IDs:  ['IA-2024-00001', 'IA-2024-00003', 'IA-2024-00005', 'IA-2024-00007', 'IA-2024-00009', 'IA-2024-00011', 'IA-2024-00013', 'IA-2024-00015', 'IA-2024-00017', 'IA-2024-00019']
Count:  10


## 2 · NumPy at speed

The single most important research-coding habit: **vectorize**. Let's prove why with the same computation two ways.

In [ ]:
speeds_list = list(range(0, 80))          # mph as a plain Python list
speeds_arr  = np.arange(0, 80)            # the same data as a NumPy array
# convert mph → km/h

In [ ]:
# SLOW — a Python-level loop
%timeit [s * 1.60934 for s in speeds_list]

4.14 µs ± 84.7 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [ ]:
# FAST — one vectorized expression over the whole array
%timeit speeds_arr * 1.60934

2.02 µs ± 55.3 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


The vectorized version is typically **50–100× faster** and shorter to read. Now the toolkit:

In [ ]:
arr = rng.normal(60, 12, size=10).round(1)      # 10 fake speed readings
print("data        :", arr)

# Broadcasting — scalar applies across the whole array, no loop
print("over by     :", (arr - 55).round(1))     # difference from a 55 mph limit

# Boolean masking — select by condition
print("speeding    :", arr[arr > 55])

# Fancy indexing — pick specific positions at once
print("1st,3rd,last:", arr[[0, 2, -1]])

# Axis operations on a 2-D array (e.g. rows = sensors, cols = time)
grid = rng.integers(0, 100, size=(3, 4))
print("col means   :", grid.mean(axis=0))       # average per time-step
print("row maxes   :", grid.max(axis=1))        # peak per sensor

data        : [63.7 47.5 69.  71.3 36.6 44.4 61.5 56.2 59.8 49.8]
over by     : [  8.7  -7.5  14.   16.3 -18.4 -10.6   6.5   1.2   4.8  -5.2]
speeding    : [63.7 69.  71.3 61.5 56.2 59.8]
1st,3rd,last: [63.7 69.  49.8]
col means   : [60.66666667 48.33333333 34.33333333 65.33333333]
row maxes   : [92 82 54]


In [ ]:
# Vectorized conditionals — replace if/else loops entirely.
speed, limit = arr, 55

# np.where → two outcomes
flag = np.where(speed > limit, "speeding", "ok")

# np.select → many outcomes (cleaner than nested np.where)
buckets = np.select(
    [speed < limit, speed < limit + 10],
    ["under",       "slightly over"],
    default="well over",          # default must share the choices' (string) dtype
)
list(zip(speed, flag, buckets))[:5]

[(np.float64(63.7), np.str_('speeding'), np.str_('slightly over')),
 (np.float64(47.5), np.str_('ok'), np.str_('under')),
 (np.float64(69.0), np.str_('speeding'), np.str_('well over')),
 (np.float64(71.3), np.str_('speeding'), np.str_('well over')),
 (np.float64(36.6), np.str_('ok'), np.str_('under'))]

In [ ]:
# Reproducible randomness — ALWAYS seed a Generator; never rely on global np.random.*
g1 = np.random.default_rng(0)
g2 = np.random.default_rng(0)
assert (g1.integers(0, 100, 5) == g2.integers(0, 100, 5)).all()   # identical every run
print("seeded RNG is reproducible ✓")

seeded RNG is reproducible ✓


**Exercise 2.** You're given `vs = rng.normal(50, 15, 1000)`. **Without any Python loop**, compute:
(a) how many readings exceed 65, and (b) the *mean* of only the readings between 40 and 60 (inclusive).
*Hint: boolean masks can be summed and used to index.*

In [ ]:
# Your turn — Exercise 2
vs = rng.normal(50, 15, 1000)
count = (vs > 65).sum()
mean = vs[(vs >= 40) & (vs <= 60)].sum() / ((vs >= 40) & (vs <= 60)).sum()

print("Count: ", count)
print("Mean: ", mean)

Count:  141
Mean:  50.356063220525286


## 3 · Pandas



In [ ]:
df = pd.read_csv("crashes_raw.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1215 entries, 0 to 1214
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   crash_id        1215 non-null   object 
 1   crash_datetime  1190 non-null   object 
 2   county          1215 non-null   object 
 3   route           1215 non-null   object 
 4   severity        1215 non-null   object 
 5   num_vehicles    1215 non-null   int64  
 6   weather         1141 non-null   object 
 7   lighting        1143 non-null   object 
 8   speed_limit     1173 non-null   float64
 9   vehicle_speed   1215 non-null   float64
 10  wrong_way       1215 non-null   bool   
 11  latitude        1174 non-null   float64
 12  longitude       1168 non-null   float64
dtypes: bool(1), float64(4), int64(1), object(7)
memory usage: 115.2+ KB


**Two things just happened that bite people constantly:**
1. `crash_datetime` came back as **`object` (string)** — CSV has no concept of dtypes. We must parse it.
2. `num_vehicles` is now **`float64`**, not int — because the `999`/missing values forced a float column.

Inspecting dtypes *before* analysing is a habit, not an afterthought.

In [ ]:
# .loc (labels) vs .iloc (integer positions) — and the chained-indexing trap
print(df.loc[0, "vehicle_speed"])     # label-based
print(df.iloc[0, 9])                  # position-based (fragile if columns move — prefer .loc)

# ❌ df[df.severity == "Fatal"]["vehicle_speed"] = 0   # chained indexing: may silently fail
# ✅ set through a single .loc instead:
demo = df.copy()
demo.loc[demo["severity"] == "Fatal", "vehicle_speed"] = np.nan
print("set via single .loc ✓")

65.0
65.0
set via single .loc ✓


In [ ]:
# Method chaining with .assign / .pipe — a readable, side-effect-free pipeline.
# Each step returns a new frame; the chain reads top-to-bottom like a recipe.
peek = (
    df
    .assign(crash_datetime=lambda d: pd.to_datetime(d["crash_datetime"], errors="coerce"))
    .assign(hour=lambda d: d["crash_datetime"].dt.hour)
    .loc[lambda d: d["vehicle_speed"].between(0, 150)]      # drop the impossible speeds inline
    [["crash_id", "county", "hour", "vehicle_speed"]]
    .head()
)
peek

,crash_id,county,hour,vehicle_speed
0,IA-2024-00000,Scott,14.0,65.0
1,IA-2024-00001,Johnson,14.0,45.0
2,IA-2024-00002,Scott,5.0,16.0
3,IA-2024-00003,Story,19.0,39.0
4,IA-2024-00004,SCOTT,16.0,41.0


In [ ]:
# groupby: agg vs transform vs apply — know which one you need.
clean_speed = df["vehicle_speed"].where(df["vehicle_speed"].between(0, 150))

# agg  → one row per group (a summary)
print("AGG — mean speed by county:")
print(df.assign(s=clean_speed).groupby("county", observed=True)["s"].agg("mean").round(1).head(), "\n")

# transform → same shape as input (broadcast the group value back to every row)
df_t = df.assign(s=clean_speed)
df_t["county_avg"] = df_t.groupby("county", observed=True)["s"].transform("mean")
print("TRANSFORM — every row gets its county average:")
print(df_t[["county", "s", "county_avg"]].head(), "\n")

# apply → arbitrary per-group function (use when agg/transform can't express it)
rng_by_county = df.assign(s=clean_speed).groupby("county", observed=True)["s"].apply(lambda x: x.max() - x.min())
print("APPLY — speed range (max-min) by county:")
print(rng_by_county.round(1).head())

AGG — mean speed by county:
county
BLACK HAWK     53.2
BLACK HAWK     55.0
Black Hawk     48.5
Black Hawk     53.7
JOHNSON        44.4
Name: s, dtype: float64 

TRANSFORM — every row gets its county average:
    county     s  county_avg
0    Scott  65.0   52.535714
1  Johnson  45.0   52.944444
2    Scott  16.0   52.535714
3    Story  39.0   52.006452
4    SCOTT  41.0   51.740741 

APPLY — speed range (max-min) by county:
county
BLACK HAWK     69.0
BLACK HAWK      0.0
Black Hawk     78.0
Black Hawk     68.0
JOHNSON        68.0
Name: s, dtype: float64


In [ ]:
# Vectorized .dt and .str — never loop to parse dates or clean text.
parsed = pd.to_datetime(df["crash_datetime"], errors="coerce")
print(".dt  → hour / weekday / is-weekend, all vectorized:")
print(pd.DataFrame({
    "hour":      parsed.dt.hour,
    "dayname":   parsed.dt.day_name(),
    "weekend":   parsed.dt.dayofweek >= 5,
}).head(), "\n")

print(".str → strip + title-case fixes 'STORY' and 'Story ' in one shot:")
print(df["county"].value_counts().head(8))                 # messy
print("→ after cleaning:")
print(df["county"].str.strip().str.title().value_counts())  # tidy

.dt  → hour / weekday / is-weekend, all vectorized:
   hour    dayname  weekend
0  14.0     Friday    False
1  14.0  Wednesday    False
2   5.0     Monday    False
3  19.0     Sunday     True
4  16.0     Friday    False 

.str → strip + title-case fixes 'STORY' and 'Story ' in one shot:
county
Scott         171
Story         157
Linn          155
Black Hawk    151
Johnson       150
Polk          146
LINN           33
BLACK HAWK     29
Name: count, dtype: int64
→ after cleaning:
county
Scott         223
Linn          210
Story         202
Black Hawk    198
Johnson       194
Polk          188
Name: count, dtype: int64


In [ ]:
# category dtype saves memory AND lets you order an ordinal variable.
before = df["county"].memory_usage(deep=True)
as_cat = df["county"].str.strip().str.title().astype("category")
after  = as_cat.memory_usage(deep=True)
print(f"county memory: {before:>7} B  →  {after:>7} B as category")

sev_order = ["No Injury", "Minor", "Major", "Fatal"]
sev = pd.Categorical(df["severity"], categories=sev_order, ordered=True)
print("ordered severity lets you compare:", (sev >= "Major").sum(), "crashes were Major or Fatal")

# Missing data is a DECISION, not a reflex. Look before you drop.
print("\nmissing per column:")
print(df.isna().sum().loc[lambda s: s > 0])

county memory:   66843 B  →     1848 B as category
ordered severity lets you compare: 134 crashes were Major or Fatal

missing per column:
crash_datetime    25
weather           74
lighting          72
speed_limit       42
latitude          41
longitude         47
dtype: int64


**Exercise 3.** Build a single chained expression that returns, for **rush-hour crashes only** (hour in 7–9 or 16–18),
the count of crashes **per county**, sorted descending.
*Hint: parse the datetime, derive `hour`, filter with `.loc[lambda d: ...]`, then `groupby(...).size()`.*

In [ ]:
# Your turn — Exercise 3
county_rush_crashes = df.assign(county=lambda d: d['county'].str.title(), hour=lambda d: pd.to_datetime(d['crash_datetime']).dt.hour).loc[lambda d: d['hour'].isin([7, 8, 9, 16, 17, 18])].groupby('county').size().sort_values(ascending=False)
county_rush_crashes

,0
county,
Scott,111
Linn,111
Black Hawk,107
Story,104
Polk,97
Johnson,96
Johnson,15
Black Hawk,13
Scott,13


## 4 · Lab — a reusable cleaning pipeline  *(today's deliverable)*

Put it together into a **typed, vectorized, documented** pipeline. Every cleaning decision is explicit — because
on Friday this goes in your repo, and in Week 4 these same decisions become your report's *Methodology · Data* section.

**No loops over rows.** Configuration lives in a `dataclass` so choices are visible and tweakable in one place.

In [ ]:
@dataclass
class CleaningConfig:
    sentinel_num_vehicles: int = 999          # code that means "unknown"
    min_speed: float = 0.0                    # speeds outside [min, max] → missing
    max_speed: float = 150.0
    title_case_cols: tuple[str, ...] = ("county", "weather", "lighting")
    drop_dupes: bool = True
    severity_order: tuple[str, ...] = ("No Injury", "Minor", "Major", "Fatal")

In [ ]:
def clean_crashes(raw: pd.DataFrame, cfg: CleaningConfig = CleaningConfig()) -> pd.DataFrame:
    """Return a cleaned copy of the raw crash frame. Pure (no mutation of the input)."""

    def parse_dates(d: pd.DataFrame) -> pd.DataFrame:
        return d.assign(crash_datetime=pd.to_datetime(d["crash_datetime"], errors="coerce"))

    def fix_sentinels(d: pd.DataFrame) -> pd.DataFrame:
        return d.assign(
            num_vehicles=d["num_vehicles"].where(d["num_vehicles"] != cfg.sentinel_num_vehicles),
            vehicle_speed=d["vehicle_speed"].where(d["vehicle_speed"].between(cfg.min_speed, cfg.max_speed)),
        )

    def tidy_text(d: pd.DataFrame) -> pd.DataFrame:
        cleaned = {c: d[c].str.strip().str.title() for c in cfg.title_case_cols if c in d}
        return d.assign(**cleaned)

    def add_features(d: pd.DataFrame) -> pd.DataFrame:
        return d.assign(
            hour=d["crash_datetime"].dt.hour,
            is_rush=d["crash_datetime"].dt.hour.isin([7, 8, 16, 17, 18]),
            severity=pd.Categorical(d["severity"], categories=cfg.severity_order, ordered=True),
        )

    out = (raw
           .pipe(parse_dates)
           .pipe(fix_sentinels)
           .pipe(tidy_text)
           .pipe(add_features))
    if cfg.drop_dupes:
        out = out.drop_duplicates(subset="crash_id").reset_index(drop=True)
    return out

clean = clean_crashes(df)
print(f"rows: {len(df)} raw → {len(clean)} clean   |   duplicates removed: {len(df) - len(clean)}")
clean.dtypes

rows: 1215 raw → 1200 clean   |   duplicates removed: 15


,0
crash_id,object
crash_datetime,datetime64[ns]
county,object
route,object
severity,category
num_vehicles,float64
weather,object
lighting,object
speed_limit,float64
vehicle_speed,float64


In [ ]:
def summarize(d: pd.DataFrame) -> pd.DataFrame:
    """Clean numeric summary table — the Monday deliverable."""
    return (d[["num_vehicles", "speed_limit", "vehicle_speed", "hour"]]
            .describe()
            .round(2)
            .T)

summary = summarize(clean)
summary

,count,mean,std,min,25%,50%,75%,max
num_vehicles,1146.0,2.43,1.13,1.0,1.0,2.0,3.0,4.0
speed_limit,1158.0,48.75,15.67,25.0,35.0,45.0,65.0,70.0
vehicle_speed,1173.0,51.36,18.01,3.0,38.0,51.0,66.0,95.0
hour,1175.0,12.37,5.96,0.0,7.0,14.0,17.0,23.0


In [ ]:
# A couple of research-relevant cuts that the cleaned frame now makes trivial:
print("Crashes by severity (note the class imbalance):")
print(clean["severity"].value_counts(), "\n")

print(f"Wrong-way crashes: {clean['wrong_way'].sum()} "
      f"({clean['wrong_way'].mean():.1%} of all crashes) — a classic rare-event target\n")

print("Mean vehicle speed by lighting condition:")
print(clean.groupby("lighting", observed=True)["vehicle_speed"].mean().round(1))

Crashes by severity (note the class imbalance):
severity
No Injury    728
Minor        341
Major        121
Fatal         10
Name: count, dtype: int64 

Wrong-way crashes: 39 (3.2% of all crashes) — a classic rare-event target

Mean vehicle speed by lighting condition:
lighting
Dark        50.9
Dawn        54.2
Daylight    51.0
Dusk        52.4
Name: vehicle_speed, dtype: float64


## 5 · Stretch

In [ ]:
# A timing decorator — wrap any function to print how long it took. functools.wraps
# keeps the original name/docstring intact.
def timed(fn: Callable) -> Callable:
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = fn(*args, **kwargs)
        print(f"{fn.__name__} took {1e3 * (time.perf_counter() - t0):.1f} ms")
        return result
    return wrapper

timed_clean = timed(clean_crashes)
_ = timed_clean(df)

clean_crashes took 17.3 ms


**Line-level profiling** (run in Colab when you want to see *which line* is slow):

```python
!pip install line_profiler -q
%load_ext line_profiler
%lprun -f clean_crashes clean_crashes(df)
```

**Light OOP** — bundle load + clean + summarize behind one object so the rest of your project just says
`CrashDataset("crashes_raw.csv").summary`:

In [ ]:
class CrashDataset:
    """Load → clean → summarize a crash CSV, with everything cached on the instance."""
    def __init__(self, path: str, cfg: CleaningConfig = CleaningConfig()):
        self.path = Path(path)
        self.cfg = cfg
        self.raw = pd.read_csv(self.path)
        self.clean = clean_crashes(self.raw, cfg)

    @property
    def summary(self) -> pd.DataFrame:
        return summarize(self.clean)

    def __repr__(self) -> str:
        return f"CrashDataset(path='{self.path.name}', rows={len(self.clean)})"

ds = CrashDataset("crashes_raw.csv")
print(ds)
ds.summary

CrashDataset(path='crashes_raw.csv', rows=1200)


,count,mean,std,min,25%,50%,75%,max
num_vehicles,1146.0,2.43,1.13,1.0,1.0,2.0,3.0,4.0
speed_limit,1158.0,48.75,15.67,25.0,35.0,45.0,65.0,70.0
vehicle_speed,1173.0,51.36,18.01,3.0,38.0,51.0,66.0,95.0
hour,1175.0,12.37,5.96,0.0,7.0,14.0,17.0,23.0


## 6 · Wrap up & commit

In [ ]:
from google.colab import files

clean.to_csv("crashes_clean.csv", index=False)
summary.to_csv("crashes_summary.csv", index=False)

files.download("crashes_clean.csv")
files.download("crashes_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Reflect (2 minutes).** Looking only at what you've seen today, write **one research question** this data raises —
something you could actually test. A few starters:

- *Are wrong-way crashes disproportionately at night, after controlling for total night-time traffic?*
- *Does the gap between vehicle speed and the posted limit predict crash severity?*
- *Which counties have crash patterns that don't look like the rest?*

---
## Homework — due before Tuesday's session

Keep everything in this notebook.

1. **Finish the three exercises** (Exercise 1, 2, and 3) left open above.
2. **Extend the pipeline.** Add a vectorized `speeding` flag to `clean_crashes` — `vehicle_speed > speed_limit + 5` — and print the fraction of crashes that are speeding-involved.
3. **Bring** your updated notebook to Tuesday. The one thing I'll check: **no `for` loop over rows, anywhere.**

---
## Completed Homework:

In [ ]:
def extended_clean_crashes(raw: pd.DataFrame, cfg: CleaningConfig = CleaningConfig()) -> pd.DataFrame:
    """Return an updated cleaned copy of the raw crash frame with speeding flag created. Pure (no mutation of the input)."""

    def parse_dates(d: pd.DataFrame) -> pd.DataFrame:
        return d.assign(crash_datetime=pd.to_datetime(d["crash_datetime"], errors="coerce"))

    def fix_sentinels(d: pd.DataFrame) -> pd.DataFrame:
        return d.assign(
            num_vehicles=d["num_vehicles"].where(d["num_vehicles"] != cfg.sentinel_num_vehicles),
            vehicle_speed=d["vehicle_speed"].where(d["vehicle_speed"].between(cfg.min_speed, cfg.max_speed)),
        )

    def tidy_text(d: pd.DataFrame) -> pd.DataFrame:
        cleaned = {c: d[c].str.strip().str.title() for c in cfg.title_case_cols if c in d}
        return d.assign(**cleaned)

    def add_features(d: pd.DataFrame) -> pd.DataFrame:
        return d.assign(
            hour=d["crash_datetime"].dt.hour,
            is_rush=d["crash_datetime"].dt.hour.isin([7, 8, 16, 17, 18]),
            severity=pd.Categorical(d["severity"], categories=cfg.severity_order, ordered=True),
            speeding=(d['vehicle_speed'] > (d['speed_limit'] + 5))
        )

    out = (raw
           .pipe(parse_dates)
           .pipe(fix_sentinels)
           .pipe(tidy_text)
           .pipe(add_features))
    if cfg.drop_dupes:
        out = out.drop_duplicates(subset="crash_id").reset_index(drop=True)
    return out

extended_clean = extended_clean_crashes(df)
print(f"rows: {len(df)} raw → {len(clean)} clean   |   duplicates removed: {len(df) - len(clean)}")
clean.dtypes
extended_clean.head()

rows: 1215 raw → 1200 clean   |   duplicates removed: 15


,crash_id,crash_datetime,county,route,severity,num_vehicles,weather,lighting,speed_limit,vehicle_speed,wrong_way,latitude,longitude,hour,is_rush,speeding
0,IA-2024-00000,2024-02-02 14:37:00,Scott,IA-1,Major,2.0,Rain,Daylight,65.0,65.0,False,41.04515,-93.62082,14.0,False,False
1,IA-2024-00001,2024-10-09 14:49:00,Johnson,I-80,Minor,2.0,Fog,Daylight,35.0,45.0,False,42.25151,-93.02592,14.0,False,True
2,IA-2024-00002,2024-08-26 05:38:00,Scott,I-35,No Injury,3.0,Clear,Dark,35.0,16.0,False,42.65609,-92.26235,5.0,False,False
3,IA-2024-00003,2024-06-09 19:35:00,Story,I-80,Minor,4.0,Clear,Daylight,25.0,39.0,False,40.92969,-92.38691,19.0,False,True
4,IA-2024-00004,2024-06-07 16:28:00,Scott,IA-210,No Injury,2.0,Rain,Dark,25.0,41.0,False,41.39147,-91.78175,16.0,True,True


In [ ]:
speed_crashes = extended_clean.loc[extended_clean['speeding']==True]
speed_crash_count = len(speed_crashes)
total_count = len(extended_clean)

print("Fraction of Speed-Involved Crashes: ", speed_crash_count, "/", total_count, " (",(speed_crash_count / total_count),")")

Fraction of Speed-Involved Crashes:  436 / 1200  ( 0.36333333333333334 )
